# Cognix Phase 3: Analysis — does TRIBE predict reading time better than LLaMA?

**The decision experiment.** Round 2 proved Cognix diverges from LLaMA. This notebook tests whether the divergence is *behaviorally useful* on Provo eye-tracking data.

**No GPU needed.** Uses cached vectors from `03_phase3_inference.ipynb` (download from Drive to local).

**The setup:**
- 55 Provo passages, each with three feature representations: TRIBE (20484-d), LLaMA (3072-d), sentence-transformer (384-d)
- Each passage has aggregate reading-time targets from 84 subjects
- Leave-one-passage-out cross-validation, ridge regression with built-in alpha selection
- Compare R² across feature sources, both raw and residualized for cloze surprisal

**The decision gate:**
- Cognix beats LLaMA on R² → the brain mapping carries useful cognitive-effort signal. Phase 4+ justified.
- Cognix ≈ LLaMA → divergence is real but not behaviorally useful for cognitive load. Re-examine.
- Cognix loses to LLaMA → brain mapping adds noise, not signal.

In [ ]:
import sys, json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from scipy import stats

# Import shared evaluation logic so the notebook and the smoke test exercise the same code
sys.path.insert(0, str(Path('..').resolve()))
from cognix.phase3_eval import (
    run_loocv, perm_null, residualize, load_features, passage_cloze, DEFAULT_ALPHAS
)

In [ ]:
# ---- Paths (local) ----
REPO = Path('..').resolve()
CHUNKS_PATH    = REPO / 'data' / 'phase3_provo_chunks.jsonl'
NORMS_PATH     = REPO / 'data' / 'provo' / 'Provo_Corpus-Predictability_Norms.csv'

# After downloading from Drive, place vectors under artifacts/
TRIBE_DIR = REPO / 'artifacts' / 'phase3_provo_pooled'
LLAMA_DIR = REPO / 'artifacts' / 'phase3_provo_llama'
ST_DIR    = REPO / 'artifacts' / 'phase3_provo_st'

for p in [CHUNKS_PATH, NORMS_PATH, TRIBE_DIR, LLAMA_DIR, ST_DIR]:
    assert p.exists(), f'Missing: {p}'
print('All paths exist. ✓')

## 1. Load chunks and feature matrices

In [ ]:
chunks = [json.loads(l) for l in open(CHUNKS_PATH)]
hashes = [c['hash'] for c in chunks]
print(f'Loaded {len(chunks)} chunks')

X_tribe = load_features(TRIBE_DIR, hashes, expected_dim=20484)
X_llama = load_features(LLAMA_DIR, hashes, expected_dim=3072)
X_st    = load_features(ST_DIR,    hashes, expected_dim=384)

print(f'X_tribe: {X_tribe.shape}')
print(f'X_llama: {X_llama.shape}')
print(f'X_st:    {X_st.shape}')

In [ ]:
# ---- Build target vectors (one scalar per passage) ----
TARGETS = ['mean_ffd_skip_excl', 'mean_fprt_skip_excl',
           'mean_trt_skip_incl', 'mean_trt_skip_excl',
           'mean_passage_total_trt']
PRIMARY_TARGET = 'mean_trt_skip_incl'  # mean per-word total reading time, captures skipping

y_dict = {t: np.array([c[t] for c in chunks], dtype=np.float32) for t in TARGETS}
for name, y in y_dict.items():
    print(f'  {name:<28} mean={y.mean():>7.1f} std={y.std():>6.1f} range={y.min():.0f}-{y.max():.0f}')

## 2. Run leave-one-out ridge regression

For each (feature source, target) combination, predict held-out passages via LOO-CV. Use RidgeCV to auto-select the regularization strength. Same procedure for all three feature sources — fair comparison.

Also compute a permutation null: shuffle the target, re-run, see the noise floor.

In [ ]:
# Use DEFAULT_ALPHAS from the shared module (np.logspace(-2, 6, 25))
# Wide range because TRIBE has 20484 features for 55 samples — needs strong regularization.
alphas = DEFAULT_ALPHAS

# Smoke test on one combo before the full grid
smoke = run_loocv(X_st, y_dict[PRIMARY_TARGET])
print(f'Smoke test (ST → {PRIMARY_TARGET}): R²={smoke["r2"]:.3f}, '
      f'Spearman={smoke["rho"]:.3f} (p={smoke["p_rho"]:.3f})')

In [ ]:
# ---- Full grid: feature source × target ----
FEATURES = {'TRIBE': X_tribe, 'LLaMA': X_llama, 'ST': X_st}

results_raw = defaultdict(dict)
for fname, X in FEATURES.items():
    for tname in TARGETS:
        results_raw[fname][tname] = run_loocv(X, y_dict[tname])

print('R² PER (FEATURE, TARGET)')
header = f'{"Target":<28}' + ''.join(f'{f:>10}' for f in FEATURES)
print(header)
print('-' * len(header))
for tname in TARGETS:
    row = f'{tname:<28}'
    for fname in FEATURES:
        row += f'{results_raw[fname][tname]["r2"]:>10.3f}'
    print(row)

print('\nSPEARMAN r PER (FEATURE, TARGET)')
print(header)
print('-' * len(header))
for tname in TARGETS:
    row = f'{tname:<28}'
    for fname in FEATURES:
        row += f'{results_raw[fname][tname]["rho"]:>10.3f}'
    print(row)

## 3. Permutation null on primary target

If LOO R² with shuffled labels can hit our observed R², the result is just noise. With 55 samples and high-dim features, the null can have meaningful right-tail mass.

In [ ]:
y_primary = y_dict[PRIMARY_TARGET]
null_results = {}
for fname, X in FEATURES.items():
    null = perm_null(X, y_primary, n_perms=200)
    obs = results_raw[fname][PRIMARY_TARGET]['r2']
    p = (null >= obs).mean()
    null_results[fname] = {'null': null, 'obs': obs, 'p': p}
    print(f'{fname}: observed R²={obs:+.3f}, null mean={null.mean():+.3f}, '
          f'null 95%ile={np.percentile(null, 95):+.3f}, p={p:.3f}')

## 4. Surprisal residualization

Provo provides cloze predictability per word from human respondents. We aggregate to mean per-passage cloze, convert to surprisal (`-log(p + ε)`), then linearly regress it out of the target. Re-running the grid on the residual asks: **what does each feature source add *beyond* word predictability?**

This addresses the "Cognix just encodes surprisal" critique.

In [ ]:
# Compute per-passage cloze using shared module function
cloze_per_passage = np.array(
    [passage_cloze(c['text_id'], NORMS_PATH) for c in chunks], dtype=np.float32
)
EPS = 1e-3
surprisal = -np.log(cloze_per_passage + EPS)

print(f'Mean cloze per passage: mean={cloze_per_passage.mean():.3f}, '
      f'range={cloze_per_passage.min():.3f}-{cloze_per_passage.max():.3f}')
print(f'Surprisal: mean={surprisal.mean():.3f}, range={surprisal.min():.3f}-{surprisal.max():.3f}')

# How well does surprisal alone predict the primary target?
rho_surp, p_surp = stats.spearmanr(surprisal, y_primary)
r_surp, _ = stats.pearsonr(surprisal, y_primary)
print(f'\nSurprisal alone vs {PRIMARY_TARGET}: Pearson r={r_surp:+.3f}, '
      f'Spearman={rho_surp:+.3f} (p={p_surp:.3f})')
print('(Positive r expected: higher surprisal -> longer reading time)')

In [ ]:
# residualize is imported from cognix.phase3_eval; sanity-check it removes surprisal effect
y_resid = residualize(y_primary, surprisal)
r_check, _ = stats.pearsonr(surprisal, y_resid)
print(f'After residualization, Pearson(surprisal, residual): {r_check:+.4f} (should be ~0)')
assert abs(r_check) < 1e-6, 'Residualization failed'
print('Residualization works. ✓')

In [ ]:
# ---- Re-run grid on residualized targets ----
results_resid = defaultdict(dict)
for fname, X in FEATURES.items():
    for tname in TARGETS:
        y_resid = residualize(y_dict[tname], surprisal)
        results_resid[fname][tname] = run_loocv(X, y_resid)

print('R² ON SURPRISAL-RESIDUALIZED TARGETS')
print('(What does each feature source add *beyond* word predictability?)')
header = f'{"Target":<28}' + ''.join(f'{f:>10}' for f in FEATURES)
print(header)
print('-' * len(header))
for tname in TARGETS:
    row = f'{tname:<28}'
    for fname in FEATURES:
        row += f'{results_resid[fname][tname]["r2"]:>10.3f}'
    print(row)

## 5. Headline plot: R² comparison on the primary target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = list(FEATURES.keys())
x = np.arange(len(labels))

for ax, results, suffix in [(axes[0], results_raw, 'raw'), (axes[1], results_resid, 'residualized')]:
    r2s = [results[f][PRIMARY_TARGET]['r2'] for f in labels]
    bars = ax.bar(x, r2s, color=['#2ecc71' if v > 0 else '#e74c3c' for v in r2s])
    ax.axhline(0, color='k', linewidth=0.5)
    if suffix == 'raw':
        # Show null 95%ile as horizontal line
        for i, f in enumerate(labels):
            null95 = np.percentile(null_results[f]['null'], 95)
            ax.plot([i-0.4, i+0.4], [null95, null95], 'k--', alpha=0.5, linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel('LOO R²')
    ax.set_title(f'Predicting {PRIMARY_TARGET}\n({suffix} target)')
    for i, v in enumerate(r2s):
        ax.text(i, v + 0.005 if v > 0 else v - 0.02, f'{v:+.3f}', ha='center', fontsize=10)
    ax.grid(True, alpha=0.2)

plt.suptitle('Phase 3: Does Cognix predict reading time better than baselines?\n'
             '(dashed line = 95th percentile of permutation null on raw target)', fontsize=12)
plt.tight_layout()
plt.savefig('phase3_r2_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Predicted vs actual scatter for each feature source on primary target ----
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
y = y_primary
for ax, fname in zip(axes, FEATURES):
    preds = results_raw[fname][PRIMARY_TARGET]['preds']
    r2 = results_raw[fname][PRIMARY_TARGET]['r2']
    rho = results_raw[fname][PRIMARY_TARGET]['rho']
    ax.scatter(y, preds, s=40, alpha=0.7, edgecolors='white', linewidth=0.5)
    lo, hi = min(y.min(), preds.min()), max(y.max(), preds.max())
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.3, label='y=x')
    ax.set_xlabel(f'Actual {PRIMARY_TARGET} (ms)')
    ax.set_ylabel('LOO predicted (ms)')
    ax.set_title(f'{fname}\nR²={r2:+.3f}  Spearman={rho:+.3f}')
    ax.legend()
    ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('phase3_pred_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Decision summary

In [ ]:
print('PHASE 3 DECISION SUMMARY')
print('=' * 65)
print(f'Primary target: {PRIMARY_TARGET}')
print(f'N passages: {len(chunks)}, eval: leave-one-out')
print()
print(f'{"Feature":<10} {"R² raw":>10} {"R² resid":>10} {"Spearman":>10} {"Perm-p":>10}')
print('-' * 60)
for f in FEATURES:
    r2_raw = results_raw[f][PRIMARY_TARGET]['r2']
    r2_res = results_resid[f][PRIMARY_TARGET]['r2']
    rho    = results_raw[f][PRIMARY_TARGET]['rho']
    p      = null_results[f]['p']
    print(f'{f:<10} {r2_raw:>+10.3f} {r2_res:>+10.3f} {rho:>+10.3f} {p:>10.3f}')

tribe_r2 = results_raw['TRIBE'][PRIMARY_TARGET]['r2']
llama_r2 = results_raw['LLaMA'][PRIMARY_TARGET]['r2']
st_r2    = results_raw['ST'][PRIMARY_TARGET]['r2']
tribe_resid = results_resid['TRIBE'][PRIMARY_TARGET]['r2']
llama_resid = results_resid['LLaMA'][PRIMARY_TARGET]['r2']

print()
print('VERDICT')
print('-' * 65)
if tribe_r2 > llama_r2 and tribe_r2 > st_r2:
    print(f'COGNIX WINS on raw target: TRIBE R²={tribe_r2:+.3f} > '
          f'LLaMA {llama_r2:+.3f}, ST {st_r2:+.3f}')
    if tribe_resid > llama_resid:
        print(f'  And after residualizing surprisal: TRIBE {tribe_resid:+.3f} > LLaMA {llama_resid:+.3f}')
        print('  -> Cognix carries cognitive-effort signal beyond word predictability.')
        print('  -> Phase 4 (region pooling) and Phase 5 (distilled model) are justified.')
    else:
        print(f'  But LLaMA wins after residualizing surprisal: {llama_resid:+.3f} vs TRIBE {tribe_resid:+.3f}')
        print('  -> Cognix\'s win on raw target is mostly explained by surprisal.')
elif tribe_r2 > llama_r2:
    print(f'COGNIX BEATS LLAMA but not ST: TRIBE {tribe_r2:+.3f}, LLaMA {llama_r2:+.3f}, ST {st_r2:+.3f}')
    print('  -> Brain mapping helps over LLaMA but a small semantic baseline still wins.')
else:
    print(f'COGNIX LOSES: TRIBE {tribe_r2:+.3f}, LLaMA {llama_r2:+.3f}, ST {st_r2:+.3f}')
    print('  -> The Round 2 divergence is real but does not translate to reading-time prediction.')
    print('  -> Re-examine Phase 4+ before building further.')
    print('  -> Possible follow-ups: try mean-centered TRIBE features, try other corpora,')
    print('     try other behavioral measures (emotional ratings, concreteness norms).')

In [ ]:
# ---- Save full results to disk ----
out = {
    'n_passages': len(chunks),
    'targets': TARGETS,
    'primary_target': PRIMARY_TARGET,
    'raw': {f: {t: {k: float(v) if not hasattr(v, 'tolist') else v.tolist()
                    for k, v in d.items()}
                for t, d in results_raw[f].items()} for f in FEATURES},
    'residualized': {f: {t: {k: float(v) if not hasattr(v, 'tolist') else v.tolist()
                             for k, v in d.items()}
                         for t, d in results_resid[f].items()} for f in FEATURES},
    'permutation_null': {f: {'observed_r2': float(null_results[f]['obs']),
                             'null_mean': float(null_results[f]['null'].mean()),
                             'null_p95': float(np.percentile(null_results[f]['null'], 95)),
                             'p_value': float(null_results[f]['p'])}
                         for f in FEATURES},
    'surprisal_alone': {'pearson_r': float(r_surp),
                        'spearman_r': float(rho_surp),
                        'p_spearman': float(p_surp)},
}
with open('phase3_results.json', 'w') as f:
    json.dump(out, f, indent=2)
print('Saved full results to phase3_results.json')